In [19]:
# importando bibliotecas
import os
import io
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# bibliotecas da atividade
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [4]:
def readFiles(path): 

    # percorre todos os diretórios e arquivos a partir do caminho dado
    for root, dirnames, filenames in os.walk(path):
        
        # para cada arquivo encontrado
        for filename in filenames:
            
            # obtém o caminho completo do arquivo
            path = os.path.join(root, filename)
            
            # inicializa variáveis para ler o corpo do email
            inBody = False
            lines = []

            # abre o arquivo com codificação latin1
            f = io.open(path, 'r', encoding='latin1')

            # lê o arquivo linha por linha
            for line in f:
                # verifica se está no corpo do email
                if inBody:
                    lines.append(line)
                elif line == '\n':
                    inBody = True
            f.close()

            # junta as linhas do corpo do email em uma única string
            message = '\n'.join(lines)
            # retorna o caminho e o corpo do email
            yield path, message

def dataFrameFromDirectory(path, classification):

    # lista de colunas e index para o DataFrame
    rows = []
    index = []

    # lê os arquivos do diretório
    for filename, message in readFiles(path):
        # adiciona uma nova linha com a mensagem e sua classificação
        rows.append({'message': message, 'class': classification})
        index.append(filename)
    
    # cria e retorna o DataFrame
    return pd.DataFrame(rows, index=index)

In [5]:
# criando um dataframe vazio
data = pd.DataFrame({'message': [], 'class': []})

# concatenando os dataframes de spam e ham ao dataframe vazio
data = pd.concat([
    data,
    dataFrameFromDirectory('arquivos/emails/spam', 'spam'),
    dataFrameFromDirectory('arquivos/emails/ham', 'ham')
], ignore_index=True)


In [6]:
# mostrando o dataframe
data

,message,class
0,NEED Health Insurance? \n\n In addition to fea...,spam
1,------=_NextPart_000_00A3_00E67D7E.B8560D80\n\...,spam
2,\n\nDear Sir or Madam\n\n\n\nIn the past you h...,spam
3,"<html>\n\n\n\n<head>\n\n<meta http-equiv=3D""Co...",spam
4,------=_NextPart_000_00D7_08E60D5B.E5437E70\n\...,spam
...,...,...
2995,Someone needs to tell the mayor about this:\n\...,ham
2996,http://www.hughes-family.org/bugzilla/show_bug...,ham
2997,"On Mon, 30 Sep 2002, Tom wrote:\n\n\n\n> If th...",ham
2998,URL: http://boingboing.net/#85494694\n\nDate: ...,ham


In [7]:
# convertendo as mensagens em uma matriz de contagem de palavras
vectorizer = CountVectorizer()
# fit_transform cria a matriz de contagem
counts = vectorizer.fit_transform(data['message'].values)

# criando e treinando o classificador Naive Bayes
classifier = MultinomialNB()
targets = data['class'].values
classifier.fit(counts, targets)

MultinomialNB()

In [8]:
# criando um exemplo para testar
examples = ['Free Viagra now!!!', 'Hi Bob, how about a game of golf tomorrow?']
# convertendo o exemplo em uma matriz de contagem
example_counts = vectorizer.transform(examples)
# fazendo a previsão
predictions = classifier.predict(example_counts)
predictions

array(['spam', 'ham'], dtype='<U4')

## Activity

Our data set is small, or our spam classifier isn't actually very good. Try running some different test emails through it and see if you get the results you expect

If you really want to challange yourself, try applying train/test to this spam classifier - see how well it can predict some subset of the ham and spam emails

In [9]:
# testando com outros exemplos
example2 = ['Come to or meeting tomorrow.', 'Get cheap meds now!!!', 'Project deadline is next week.', 'Come home Now!!!']
example2_counts = vectorizer.transform(example2)

predictions2 = classifier.predict(example2_counts)
predictions2

array(['ham', 'ham', 'ham', 'ham'], dtype='<U4')

O resultado não foi tão bom, pois o segundo poderia facilmente ser um spam

Agora tentando aplicar treino e teste

In [ ]:
# utilizando a função train_test_split para dividir os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    data['message'].values,
    data['class'].values,
    test_size=0.25,
    random_state=42
)

In [ ]:
# convertendo as mensagens de treino e teste em matrizes de contagem de palavras
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

In [12]:
# treinando o classificador Naive Bayes com os dados de treino
classifier = MultinomialNB()
classifier.fit(X_train_counts, y_train)

MultinomialNB()

In [14]:
# fazendo previsões com os dados de teste
predictions = classifier.predict(X_test_counts)

In [ ]:
# vendo a acurácia do modelo
accuracy_score(y_test, predictions)

0.9426666666666667